# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
## Method Choice and Why

For this assignment, I selected the Decision Tree Classifier.

The Refresh / Content Opportunity Scoring lane focuses on identifying webpages that are likely to require content updates based on observable page-level metrics. A Decision Tree is appropriate because it produces human-readable decision rules that explain why a page is classified as declining or not declining.

Compared to more complex ensemble methods, a Decision Tree provides good interpretability, making it easier for a content team to understand which signals influence recommendations. This aligns with the objective of creating decision-support rather than a black-box prediction system.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I used a standard train-test split with 80% of the data used for training and 20% reserved for testing.

The split was performed using a fixed random state to ensure reproducibility.

The same target variable and feature set were used consistently throughout the experiment so that the model could be compared fairly against the earlier baseline. The test data remained unseen during training, providing an honest estimate of model performance.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create target
df["is_declining"] = (
    df["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

# Features
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count"
]

X = (
    df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

y = df["is_declining"]

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Baseline
baseline_prediction = np.zeros(len(y_test))

baseline_accuracy = accuracy_score(y_test, baseline_prediction)

# Decision Tree
model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

# Metrics
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

comparison = pd.DataFrame({
    "Model":[
        "Baseline",
        "Decision Tree"
    ],
    "Accuracy":[
        round(baseline_accuracy,3),
        round(accuracy,3)
    ]
})

print(comparison)

print("\nDecision Tree Metrics")

print("Accuracy :", round(accuracy,3))
print("Precision:", round(precision,3))
print("Recall   :", round(recall,3))
print("F1 Score :", round(f1,3))

           Model  Accuracy
0       Baseline     0.458
1  Decision Tree     0.637

Decision Tree Metrics
Accuracy : 0.637
Precision: 0.644
Recall   : 0.738
F1 Score : 0.688


In [5]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

                  Feature  Importance
2         impressions_90d    0.582405
0        content_age_days    0.244074
3            avg_position    0.086973
4                     ctr    0.061020
1  days_since_last_update    0.025327
5              word_count    0.000202


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*


The Decision Tree achieved better performance than the simple baseline while remaining easy to interpret.

Some webpages were incorrectly classified because several pages have similar engagement characteristics even though their trend direction differs. This indicates that page performance depends on multiple interacting factors rather than a single metric.

Feature importance suggests that metrics such as impressions, average position, content age, and days since the last update contribute most to the model's decisions.

The model should therefore be considered a decision-support tool that helps prioritize pages for review rather than making final business decisions automatically.

## Self-check

Before you submit, confirm each line honestly:

- [✅ ] Every section above is filled — markdown thinking AND the code that backs it
- [✅ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅ ] My claims use careful words: observed, measured, directional, decision-support
- [✅ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.